[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/03_simulation.ipynb)

# Tutorial 3: Simulation

Understand `mj_step`, `mj_forward`, timesteps, and the simulation pipeline.

In [ ]:
!pip install -q mujoco mediapy matplotlib

In [ ]:
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media

## The Simulation Loop

MuJoCo's main function is `mj_step(model, data)` which advances the state by one timestep:

$$x_{t+h} = f(x_t, u_t)$$

where $x = (q, \dot{q})$ is the state and $u$ is the control.

In [ ]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81" timestep="0.001"/>
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body name="ball" pos="0 0 1">
      <joint type="free"/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

print(f"Timestep: {model.opt.timestep}s")
print(f"Initial height: {data.qpos[2]:.3f}")

# Simulate for 1 second
times, heights = [], []
while data.time < 1.0:
    mujoco.mj_step(model, data)
    times.append(data.time)
    heights.append(data.qpos[2])

plt.figure(figsize=(10, 4))
plt.plot(times, heights)
plt.xlabel('Time (s)'); plt.ylabel('Height (m)')
plt.title('Ball dropping and bouncing')
plt.grid(True); plt.show()

## `mj_step` vs `mj_forward`

- **`mj_step`**: Advances time. Computes forces → integrates → updates state.
- **`mj_forward`**: Computes all derived quantities **without** advancing time.

Use `mj_forward` when you need positions/velocities after setting `qpos`/`qvel` manually.

In [ ]:
# Reset and manually set position
mujoco.mj_resetData(model, data)
data.qpos[2] = 5.0  # set height to 5m

# mj_forward propagates derived quantities
mujoco.mj_forward(model, data)
print(f"Ball position: {data.qpos[:3]}")
print(f"Ball velocity: {data.qvel[:3]}")
print(f"Time did NOT advance: {data.time}")

## Timestep and Stability

Smaller timesteps = more accurate but slower. MuJoCo's default is 2ms.
Let's compare:

In [ ]:
plt.figure(figsize=(10, 4))

for dt in [0.001, 0.005, 0.01]:
    model_dt = mujoco.MjModel.from_xml_string(xml)
    model_dt.opt.timestep = dt
    data_dt = mujoco.MjData(model_dt)

    times, heights = [], []
    while data_dt.time < 1.0:
        mujoco.mj_step(model_dt, data_dt)
        times.append(data_dt.time)
        heights.append(data_dt.qpos[2])

    plt.plot(times, heights, label=f'dt={dt}s ({len(times)} steps)')

plt.xlabel('Time (s)'); plt.ylabel('Height (m)')
plt.title('Effect of timestep on simulation')
plt.legend(); plt.grid(True); plt.show()

## Energy & Momentum Tracking

In [ ]:
model_e = mujoco.MjModel.from_xml_string(xml)
model_e.opt.timestep = 0.001
data_e = mujoco.MjData(model_e)

# Disable contacts to see energy conservation
model_e.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_CONTACT

times, kinetic, potential = [], [], []
while data_e.time < 0.45:  # before hitting ground
    mujoco.mj_step(model_e, data_e)
    times.append(data_e.time)
    # KE = 0.5 * m * v^2
    v = np.linalg.norm(data_e.qvel[:3])
    kinetic.append(0.5 * 1.0 * v**2)
    # PE = m * g * h
    potential.append(1.0 * 9.81 * data_e.qpos[2])

total = np.array(kinetic) + np.array(potential)

plt.figure(figsize=(10, 4))
plt.plot(times, kinetic, label='Kinetic')
plt.plot(times, potential, label='Potential')
plt.plot(times, total, 'k--', label='Total')
plt.xlabel('Time (s)'); plt.ylabel('Energy (J)')
plt.title('Energy conservation (no contacts)')
plt.legend(); plt.grid(True); plt.show()
print(f"Energy drift: {(total[-1] - total[0])/total[0]*100:.6f}%")